In [1]:
import numpy as np

def cholesky_decomposition(A):
    """
    Performs Cholesky decomposition on a symmetric, positive-definite
    matrix A such that A = L * L.T, where L is a lower-triangular matrix.

    Args:
        A (np.ndarray): A symmetric, positive-definite square matrix.

    Returns:
        np.ndarray: The lower-triangular matrix L.

    Raises:
        ValueError: If the matrix is not square, not symmetric, or
                    not positive-definite.
    """

    # 1. Prerequisite Checks
    n, m = A.shape
    if n != m:
        raise ValueError("Matrix must be square.")

    # Check for symmetry. np.allclose handles floating-point inaccuracies.
    if not np.allclose(A, A.T):
        raise ValueError("Matrix must be symmetric.")

    # 2. Initialization
    # Create an empty (zero) matrix for L
    L = np.zeros_like(A, dtype=float)

    # 3. Main Algorithm (Cholesky–Banachiewicz algorithm)
    # We compute L element by element, row by row.
    for i in range(n):
        for j in range(i + 1):  # For each element up to and including the diagonal

            # This is the dot product of two vectors:
            # L[i, :j] and L[j, :j]
            s = 0.0
            for k in range(j):
                s += L[i, k] * L[j, k]

            # 4. Handle Diagonal vs. Off-Diagonal
            if i == j:
                # Diagonal element: L[i, i]
                # Formula: L[i, i] = sqrt( A[i, i] - sum(L[i, k]^2 for k < i) )

                val_under_sqrt = A[i, i] - s

                # Check for positive-definiteness
                # If val_under_sqrt is negative or zero, we can't take the
                # square root, and the matrix is not positive-definite.
                if val_under_sqrt < 1e-9: # Use a small epsilon for floats
                    raise ValueError(f"Matrix is not positive-definite. "
                                     f"Failed at diagonal element {i}.")

                L[i, j] = np.sqrt(val_under_sqrt)
            else:
                # Off-diagonal element: L[i, j]
                # Formula: L[i, j] = (1 / L[j, j]) * (A[i, j] - sum(L[i, k] * L[j, k] for k < j))

                # We can safely divide by L[j, j] because it's a diagonal
                # element that was already computed and checked.
                L[i, j] = (A[i, j] - s) / L[j, j]

    return L

if __name__ == "__main__":
    # Set print options for numpy for cleaner output
    np.set_printoptions(precision=4, suppress=True)

    # --- Example 1: A valid 3x3 matrix ---
    # This matrix is symmetric and positive-definite.
    A1 = np.array([
        [4, 12, -16],
        [12, 37, -43],
        [-16, -43, 98]
    ], dtype=float)

    print("--- Example 1 (Valid Matrix) ---")
    print("Original Matrix A:\n", A1)

    try:
        # Our implementation
        L_manual = cholesky_decomposition(A1)
        print("\nOur L:\n", L_manual)

        # NumPy's built-in implementation for comparison
        L_numpy = np.linalg.cholesky(A1)
        print("\nNumPy's L:\n", L_numpy)

        # Check: Does L * L.T = A?
        # The @ operator is for matrix multiplication
        A_reconstructed = L_manual @ L_manual.T
        print("\nL @ L.T (Reconstructed A):\n", A_reconstructed)

        print("\nSuccess:", np.allclose(A1, A_reconstructed))

    except ValueError as e:
        print("\nFailed:", e)

    print("-" * 30)

    # --- Example 2: A non-positive-definite matrix ---
    A2 = np.array([
        [1, 2],
        [2, 1]
    ], dtype=float)

    print("\n--- Example 2 (Not Positive-Definite) ---")
    print("Original Matrix A:\n", A2)

    try:
        L = cholesky_decomposition(A2)
        print("\nOur L:\n", L)
    except ValueError as e:
        print("\nFailed as expected:")
        print(e)

    print("-" * 30)

--- Example 1 (Valid Matrix) ---
Original Matrix A:
 [[  4.  12. -16.]
 [ 12.  37. -43.]
 [-16. -43.  98.]]

Our L:
 [[ 2.  0.  0.]
 [ 6.  1.  0.]
 [-8.  5.  3.]]

NumPy's L:
 [[ 2.  0.  0.]
 [ 6.  1.  0.]
 [-8.  5.  3.]]

L @ L.T (Reconstructed A):
 [[  4.  12. -16.]
 [ 12.  37. -43.]
 [-16. -43.  98.]]

Success: True
------------------------------

--- Example 2 (Not Positive-Definite) ---
Original Matrix A:
 [[1. 2.]
 [2. 1.]]

Failed as expected:
Matrix is not positive-definite. Failed at diagonal element 1.
------------------------------
